# Quantum Twin -- Quickstart

This notebook is a thin **consumer** of the `quantum_twin` library -- it
installs the package from this repository and calls into it, the same
way any other user of the library would. It does NOT redefine or
recreate any of the library's source code inline (unlike this project's
pre-v4.0 notebooks, which used `%%writefile` cells to reconstruct the
entire package from scratch on every run): the library under
`src/quantum_twin/` is now the single source of truth, versioned,
tested, and documented on its own -- see [`docs/`](../docs/) for the
full documentation site, [`experiments/`](../experiments/) for
standalone experiment scripts, and [`tests/`](../tests/) for the test
suite.

**Run this notebook from within a clone of the repository** (e.g. open
it via `notebooks/quickstart.ipynb` after `git clone`, or upload the
whole repository to Colab) -- the install cell below assumes the parent
directory is the repository root.


## 0. Install the package

In [ ]:
# Installs quantum_twin (and its dependencies) in editable mode from the
# repository this notebook lives in. Run once per fresh runtime.
%pip install -q -e ..


In [ ]:
import quantum_twin

print(f"quantum_twin version: {quantum_twin.__version__}")


## 1. Quickstart: train EdgeLSTM and compare against blind purification

The smallest complete example -- generate the synthetic channel, train
`EdgeLSTM + CS_MSELoss`, and check whether the predictive controller
beats unconditional (blind) purification. See
[`docs/getting-started/quickstart.md`](../docs/getting-started/quickstart.md)
for the same walkthrough with full explanations.

In [ ]:
import torch

from quantum_twin.channel_simulator import WDMChannelSimulator
from quantum_twin.cli import get_device
from quantum_twin.config import QuantumConfig, SimConfig, TrainConfig
from quantum_twin.models import EdgeLSTM, train_edge_lstm
from quantum_twin.orchestrator import DigitalTwinOrchestrator
from quantum_twin.quantum_node import QuantumRepeaterNode
from quantum_twin.reproducibility import set_full_determinism

DEVICE = get_device()
print(f"Selected PyTorch device: {DEVICE}")

set_full_determinism(seed=42)

sim_cfg = SimConfig()
train_cfg = TrainConfig()
quantum_cfg = QuantumConfig()


In [ ]:
wdm_sim = WDMChannelSimulator(n_steps=sim_cfg.n_steps, dt=sim_cfg.dt, seed=sim_cfg.seed)
df = wdm_sim.generate_dataset()
X_train, y_train, X_test, y_test, _scaler = wdm_sim.preprocess(
    df, window_size=sim_cfg.window_size, test_size=sim_cfg.test_size,
)
X_train, y_train = X_train.to(DEVICE), y_train.to(DEVICE)
X_test, y_test = X_test.to(DEVICE), y_test.to(DEVICE)

print(f"Training windows: {len(X_train)} | Test windows: {len(X_test)}")


In [ ]:
model = EdgeLSTM(input_size=2, hidden_size=train_cfg.hidden_size).to(DEVICE)
model = train_edge_lstm(
    model, X_train, y_train, threshold=train_cfg.threshold,
    lambda_penalty=10.0, lambda_fn=train_cfg.lambda_fn,
    discard_penalty_weight=train_cfg.discard_penalty_weight,
    max_discard_rate=train_cfg.max_discard_rate,
    epochs=train_cfg.epochs, lr=train_cfg.lr, device=DEVICE, seed=42,
)


In [ ]:
quantum_node = QuantumRepeaterNode(T1=quantum_cfg.T1, T2=quantum_cfg.T2, depol_prob=quantum_cfg.depol_prob,
                                    shots=quantum_cfg.shots, seed=quantum_cfg.seed)
orchestrator = DigitalTwinOrchestrator(model=model, quantum_node=quantum_node,
                                         threshold=train_cfg.threshold, device=DEVICE)
intelligent_metrics = orchestrator.run_intelligent(X_test, y_test)

baseline_node = QuantumRepeaterNode(T1=quantum_cfg.T1, T2=quantum_cfg.T2, depol_prob=quantum_cfg.depol_prob,
                                     shots=quantum_cfg.shots, seed=quantum_cfg.seed)
baseline_orchestrator = DigitalTwinOrchestrator(model=None, quantum_node=baseline_node,
                                                  threshold=train_cfg.threshold, device=DEVICE)
blind_metrics = baseline_orchestrator.run_blind_baseline(X_test, y_test)

print(f"Intelligent controller: {intelligent_metrics['useful_pairs']} useful pairs "
      f"from {intelligent_metrics['attempted']} attempts "
      f"({intelligent_metrics['useful_pairs']/max(intelligent_metrics['attempted'],1)*100:.1f}% yield)")
print(f"Blind baseline:         {blind_metrics['useful_pairs']} useful pairs "
      f"from {blind_metrics['attempted']} attempts "
      f"({blind_metrics['useful_pairs']/max(blind_metrics['attempted'],1)*100:.1f}% yield)")


## 2. Full experiments

Every experiment type below is one function call (library usage) or one
script away (`python experiments/run_*.py` from a terminal) -- see
[`docs/guides/experiments.md`](../docs/guides/experiments.md) for the
full reference. Each cell here runs the SAME functions the standalone
scripts under [`experiments/`](../experiments/) call; use whichever is
more convenient for your workflow.

### 2.1 Pareto sweep over `lambda_penalty` (multi-seed)

Equivalent to `python experiments/run_pareto_sweep.py`.

In [ ]:
from quantum_twin.config import SweepConfig
from quantum_twin.pareto_sweep import run_pareto_sweep

sweep_cfg = SweepConfig(lambda_values=[1.0, 2.0, 5.0, 10.0, 20.0, 50.0], seeds=list(range(42, 52)))

results_df, baseline_metrics, per_seed_results = run_pareto_sweep(
    sweep_cfg.lambda_values, X_train, y_train, X_test, y_test, device=DEVICE,
    threshold=train_cfg.threshold, epochs=train_cfg.epochs, lr=train_cfg.lr,
    hidden_size=train_cfg.hidden_size, T1=quantum_cfg.T1, T2=quantum_cfg.T2,
    depol_prob=quantum_cfg.depol_prob, shots=quantum_cfg.shots, quantum_seed=quantum_cfg.seed,
    lambda_fn=train_cfg.lambda_fn, discard_penalty_weight=train_cfg.discard_penalty_weight,
    max_discard_rate=train_cfg.max_discard_rate, seeds=sweep_cfg.seeds,
)
results_df


In [ ]:
from quantum_twin import plotting

fig = plotting.plot_pareto_frontier(results_df, metric_cols=("QPU Yield (%)", "MAE"))


### 2.2 Cross-architecture comparison + statistical significance

Equivalent to `python experiments/run_model_comparison.py`.

In [ ]:
from quantum_twin.config import BaselineConfig, ComparisonConfig, EnergyConfig
from quantum_twin.model_comparison import run_model_comparison
from quantum_twin.statistics_tests import compare_models_statistically

baseline_cfg = BaselineConfig()
energy_cfg = EnergyConfig()
comparison_cfg = ComparisonConfig(representative_lambda=10.0, seeds=list(range(42, 52)))

(comp_results_df, comp_baseline_metrics, decision_matrix_df,
 per_model_seed_results, sensitivity_results) = run_model_comparison(
    X_train, y_train, X_test, y_test, device=DEVICE,
    train_cfg=train_cfg, quantum_cfg=quantum_cfg, baseline_cfg=baseline_cfg,
    energy_cfg=energy_cfg, comparison_cfg=comparison_cfg,
)
comp_results_df


In [ ]:
decision_matrix_df  # Rank 1 = recommended model (Oracle is excluded -- see its docstring)


In [ ]:
significance_df = compare_models_statistically(
    per_model_seed_results, metric_key="qpu_yield_pct", reference_model="EdgeLSTM+CS-MSE",
)
significance_df


In [ ]:
fig = plotting.plot_significance_forest(significance_df)


### 2.3 2x2 factorial ablation study

Equivalent to `python experiments/run_ablation.py`.

In [ ]:
from quantum_twin.ablation import run_ablation_study
from quantum_twin.config import AblationConfig

ablation_cfg = AblationConfig(representative_lambda=10.0, seeds=list(range(42, 52)))

ablation_results_df, decomposition_df, ablation_baseline_metrics, per_cell_seed_results = run_ablation_study(
    X_train, y_train, X_test, y_test, device=DEVICE,
    train_cfg=train_cfg, quantum_cfg=quantum_cfg, ablation_cfg=ablation_cfg,
)
ablation_results_df


In [ ]:
decomposition_df


In [ ]:
for metric in decomposition_df["Metric"]:
    fig = plotting.plot_ablation_interaction(decomposition_df, metric)


### 2.4 Walk-forward (rolling-origin) temporal cross-validation

Equivalent to `python experiments/run_walk_forward.py`.

In [ ]:
from quantum_twin.config import WalkForwardConfig
from quantum_twin.walk_forward import run_walk_forward_evaluation

wf_cfg = WalkForwardConfig(n_splits=5, test_size=150, min_train_size=300)

# test_size=0.0: every windowed sample lands in the "train" output;
# run_walk_forward_evaluation performs its own chronological splitting internally.
X_full, y_full, _unused_X, _unused_y, _scaler = wdm_sim.preprocess(
    df, window_size=sim_cfg.window_size, test_size=0.0,
)

fold_df, wf_summary_df, wf_splits = run_walk_forward_evaluation(
    X_full, y_full, DEVICE, train_cfg=train_cfg, quantum_cfg=quantum_cfg, wf_cfg=wf_cfg,
)
wf_summary_df


In [ ]:
fig = plotting.plot_walk_forward_folds(fold_df, wf_summary_df, metric="qpu_yield_pct")


## 3. Recording this run

Local tracking (always available, zero setup) writes every config/table/
figure to a timestamped directory. See
[`docs/guides/experiment-tracking.md`](../docs/guides/experiment-tracking.md)
for the MLflow-backed alternative (`quantum_twin.mlops.MLflowTracker`,
optional, degrades gracefully if `mlflow` isn't installed).

In [ ]:
from quantum_twin.experiment_tracking import track_pareto_sweep_experiment

exp = track_pareto_sweep_experiment(
    "pareto_sweep_notebook_run", results_df, baseline_metrics,
    sim_cfg, train_cfg, quantum_cfg, sweep_cfg, DEVICE,
)
print(f"Artifacts saved to: {exp.dir}")


## Next steps

- Full architecture and design rationale: [`docs/architecture.md`](../docs/architecture.md)
- Every experiment type, in depth: [`docs/guides/experiments.md`](../docs/guides/experiments.md)
- Complete API reference: [`docs/api/index.md`](../docs/api/index.md) (or `mkdocs serve` for the live site)
- Contributing / running the test suite: [`docs/contributing.md`](../docs/contributing.md)
